In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mga AI transpiler pass
Ang mga AI-powered transpiler pass ay mga pass na gumagana bilang kapalit ng mga "tradisyonal" na Qiskit pass para sa ilang gawain sa transpiling. Kadalasan, mas maganda ang kanilang mga resulta kumpara sa mga umiiral na heuristic na algorithm (tulad ng mas mababang depth at CNOT count), at mas mabilis pa rin sila kaysa sa mga optimization algorithm tulad ng Boolean satisfiability solver. Ang mga AI transpiler pass ay maaaring tumakbo sa iyong lokal na environment o sa cloud gamit ang Qiskit Transpiler Service kung ikaw ay bahagi ng IBM Quantum&reg; Premium Plan, Flex Plan, o On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan.

> **Note:** Ang mga AI-powered transpiler pass ay nasa beta release status pa, at maaaring magbago.
>     Kung mayroon kang feedback o nais makipag-ugnayan sa development team, gamitin ang [Qiskit Slack Workspace channel](https://qiskit.slack.com/archives/C06KF8YHUAU) na ito.

Ang mga sumusunod na pass ay kasalukuyang available:

**Mga Routing pass**
 - `AIRouting`: Pagpili ng layout at pag-route ng Circuit

**Mga Circuit synthesis pass**
 - `AICliffordSynthesis`: Clifford circuit synthesis
 - `AILinearFunctionSynthesis`: Linear function circuit synthesis
 - `AIPermutationSynthesis`: Permutation circuit synthesis
 - `AIPauliNetworkSynthesis`: Pauli Network circuit synthesis

Para gamitin ang mga AI transpiler pass, [i-install muna ang `qiskit-ibm-transpiler` package](/guides/qiskit-transpiler-service#install-transpiler-service). Bisitahin ang [qiskit-ibm-transpiler API documentation](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler) para sa mas detalyadong impormasyon tungkol sa iba't ibang available na opsyon.

## Patakbuhin ang mga AI transpiler pass nang lokal o sa cloud
Kung nais mong gamitin ang mga AI-powered transpiler pass sa iyong lokal na environment nang libre, i-install ang `qiskit-ibm-transpiler` na may dagdag na dependencies tulad ng sumusunod:

In [1]:
from qiskit.transpiler import PassManager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_runtime import QiskitRuntimeService

backend = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=backend,
            optimization_level=2,
            layout_mode="optimize",
            local_mode=True,
        )
    ]
)


circuit = efficient_su2(101, entanglement="circular", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Kung walang mga dagdag na dependencies na ito, ang mga AI-powered transpiler pass ay tatakbo sa cloud sa pamamagitan ng Qiskit Transpiler Service (available lamang para sa mga gumagamit ng IBM Quantum Premium Plan, Flex Plan, o On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan). Pagkatapos i-install ang mga karagdagang dependencies, ang default na paraan para patakbuhin ang mga AI-powered transpiler pass ay ang gamitin ang iyong lokal na makina.

## AI routing pass
Ang `AIRouting` pass ay gumaganap bilang layout stage at routing stage. Maaari itong gamitin sa loob ng isang `PassManager` tulad ng sumusunod:

In [ ]:
from qiskit.transpiler import PassManager

from qiskit_ibm_transpiler.ai.routing import AIRouting
from qiskit_ibm_transpiler.ai.synthesis import AILinearFunctionSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectLinearFunctions
from qiskit_ibm_transpiler.ai.synthesis import AIPauliNetworkSynthesis
from qiskit_ibm_transpiler.ai.collection import CollectPauliNetworks
from qiskit.circuit.library import efficient_su2

ibm_torino = QiskitRuntimeService().backend("ibm_torino")
ai_passmanager = PassManager(
    [
        AIRouting(
            backend=ibm_torino,
            optimization_level=3,
            layout_mode="optimize",
            local_mode=True,
        ),  # Route circuit
        CollectLinearFunctions(),  # Collect Linear Function blocks
        AILinearFunctionSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Linear Function blocks
        CollectPauliNetworks(),  # Collect Pauli Networks blocks
        AIPauliNetworkSynthesis(
            backend=ibm_torino, local_mode=True
        ),  # Re-synthesize Pauli Network blocks.
    ]
)

circuit = efficient_su2(10, entanglement="full", reps=1)

transpiled_circuit = ai_passmanager.run(circuit)

Dito, tinutukoy ng `backend` kung aling coupling map ang gagamitin para sa routing, tinutukoy ng `optimization_level` (1, 2, o 3) kung gaano karaming computational effort ang gagamitin sa proseso (mas mataas ay karaniwang nagbibigay ng mas magandang resulta ngunit mas matagal), at tinutukoy ng `layout_mode` kung paano hawakan ang pagpili ng layout.
Kasama sa `layout_mode` ang mga sumusunod na opsyon:

- `keep`: Iginagalang nito ang layout na itinakda ng mga nakaraang transpiler pass (o gumagamit ng trivial layout kung hindi pa naitakda). Karaniwang ginagamit lamang ito kapag kailangang patakbuhin ang Circuit sa mga partikular na qubit ng device. Kadalasan, nagbibigay ito ng mas masamang resulta dahil mas limitado ang pagkakataon para sa optimization.
- `improve`: Ginagamit nito ang layout na itinakda ng mga nakaraang transpiler pass bilang panimula. Kapaki-pakinabang ito kapag mayroon kang magandang paunang hulà para sa layout; halimbawa, para sa mga Circuit na ginawa sa paraang halos sumusunod sa coupling map ng device. Kapaki-pakinabang din ito kung nais mong subukan ang iba pang partikular na layout pass kasama ang `AIRouting` pass.
- `optimize`: Ito ang default na mode. Pinakaepektibo ito para sa mga pangkalahatang Circuit kung saan maaaring wala kang magandang hulà sa layout. Binabalewala ng mode na ito ang mga nakaraang pagpili ng layout.
- `local_mode`: Tinutukoy ng flag na ito kung saan tatakbo ang `AIRouting` pass. Kung `False`, ang `AIRouting` ay tatakbo nang remote sa pamamagitan ng Qiskit Transpiler Service. Kung `True`, susubukan ng package na patakbuhin ang pass sa iyong lokal na environment na may fallback sa cloud mode kung hindi mahanap ang mga kinakailangang dependencies.

## Mga AI circuit synthesis pass
Ang mga AI circuit synthesis pass ay nagbibigay-daan sa iyo na i-optimize ang mga bahagi ng iba't ibang uri ng Circuit ([Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford), [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction), [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation), Pauli Network) sa pamamagitan ng muling pag-synthesize sa kanila. Ang isang tipikal na paraan para gamitin ang synthesis pass ay tulad ng sumusunod:

In [ ]:
from qiskit_ibm_transpiler import generate_ai_pass_manager
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import QiskitRuntimeService


backend = QiskitRuntimeService().backend("ibm_torino")
torino_coupling_map = backend.coupling_map


su2_circuit = efficient_su2(101, entanglement="circular", reps=1)

ai_transpiler_pass_manager = generate_ai_pass_manager(
    coupling_map=torino_coupling_map,
    ai_optimization_level=3,
    optimization_level=3,
    ai_layout_mode="optimize",
)

ai_su2_transpiled_circuit = ai_transpiler_pass_manager.run(su2_circuit)

Iginagalang ng synthesis ang coupling map ng device: maaari itong patakbuhin nang ligtas pagkatapos ng iba pang routing pass nang hindi naaabala ang Circuit, kaya ang pangkalahatang Circuit ay susunod pa rin sa mga paghihigpit ng device. Sa default, papalitan ng synthesis ang orihinal na sub-circuit lamang kung mas maganda ang synthesized sub-circuit kaysa sa orihinal (kasalukuyang tinitingnan lamang ang CNOT count), ngunit maaari itong pilitin na palaging palitan ang Circuit sa pamamagitan ng pagtatakda ng `replace_only_if_better=False`.

Ang mga sumusunod na synthesis pass ay available mula sa `qiskit_ibm_transpiler.ai.synthesis`:

- *AICliffordSynthesis*: Synthesis para sa mga [Clifford](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.Clifford) circuit (mga bloke ng `H`, `S`, at `CX` gate). Kasalukuyang hanggang siyam na qubit block.
- *AILinearFunctionSynthesis*: Synthesis para sa mga [Linear Function](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.LinearFunction) circuit (mga bloke ng `CX` at `SWAP` gate). Kasalukuyang hanggang siyam na qubit block.
- *AIPermutationSynthesis*: Synthesis para sa mga [Permutation](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.Permutation#permutation) circuit (mga bloke ng `SWAP` gate). Kasalukuyang available para sa 65, 33, at 27 na qubit block.
- *AIPauliNetworkSynthesis*: Synthesis para sa mga Pauli Network circuit (mga bloke ng `H`, `S`, `SX`, `CX`, `RX`, `RY` at `RZ` gate). Kasalukuyang hanggang anim na qubit block.

Inaasahan naming unti-unting palakihin ang suportadong laki ng mga bloke.

Lahat ng pass ay gumagamit ng thread pool para magpadala ng ilang kahilingan nang sabay-sabay. Sa default, ang bilang ng maximum na thread ay katumbas ng bilang ng mga core kasama ang apat (default na halaga para sa `ThreadPoolExecutor` Python object). Gayunpaman, maaari kang magtakda ng sariling halaga gamit ang `max_threads` na argumento sa instantiation ng pass. Halimbawa, ang sumusunod na linya ay nag-instantiate ng `AILinearFunctionSynthesis` pass, na nagpapahintulot dito na gumamit ng maximum na 20 thread.